# Worked example — one figure, both outputs

This notebook exists to prove the wiring on a fresh clone, and to be the shortest possible
answer to "how do I add a figure?". It uses synthetic data so it runs with no analysis
stack and no credentials. Delete it once your own first notebook replaces it.

What it demonstrates:

1. **One figure object, two outputs.** `save_fig` writes the PNG that the document
   compiles *and* an interactive HTML export, from the same object, so the two cannot
   disagree.
2. **The config decides, not the source.** Whether anything is written is
   `figures_config.toml`'s call — no `SAVE = False` constant to flip, no commenting a
   call out.
3. **Re-running is safe.** The `\begin{figure}` block is appended only if the document
   does not reference the figure yet, so this notebook can run a hundred times without
   growing the chapter.

## Setup

`notebook_savers` binds the routing — which chapter, which notebook, which `.tex` file —
once, so individual calls stay about the figure rather than about where it goes.

In [ ]:
import sys
from pathlib import Path

# Make notebooks/setup_notebook.py importable from any depth. A kernel's working directory
# depends on what launched it -- the notebook's own directory under JupyterLab and VS Code,
# the project root under PyCharm -- so walk up and look for it rather than assuming.
for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "notebooks" / "setup_notebook.py").exists():
        sys.path.insert(0, str(_candidate / "notebooks"))
        break
else:
    raise FileNotFoundError(f"notebooks/setup_notebook.py not found above {Path.cwd()}")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from setup_notebook import ROOT  # chdir's to the repo root; registers the plotly template
from doc_analysis import notebook_savers, figure_size
from doc_analysis.theme import CATEGORICAL

# Where this notebook's artefacts belong in the document repo. CHAPTER is a path under
# $DOC_REPO/Chapters; TEX names the file that receives a reference, which is only
# required in chapters holding several .tex files (elsewhere the single candidate is
# found automatically, and an ambiguous one raises rather than guessing).
CHAPTER = "Example"
NOTEBOOK = "example_figure.ipynb"
TEX = "example.tex"

save_fig, save_table = notebook_savers(chapter=CHAPTER, notebook=NOTEBOOK, tex=TEX)
print("repo root:", ROOT)

## Some data

Synthetic, so this notebook has no dependencies beyond the tooling itself. Replace with
whatever your analysis produces.

In [ ]:
rng = np.random.default_rng(0)
n = 42

reference = rng.uniform(10, 90, n)
measured = reference * rng.normal(1.0, 0.08, n) + rng.normal(0, 2.0, n)

data = pd.DataFrame({
    "case": [f"case-{i:02d}" for i in range(1, n + 1)],
    "reference": reference,
    "measured": measured,
    "group": rng.choice(["a", "b"], n),
})
data["residual"] = data["measured"] - data["reference"]
data.head()

## The figure

Three things here are conventions rather than taste, and each one is load-bearing:

- **An explicit `hovertemplate`** carrying the identifying field. The interactive export
  is the whole reason for the dual output; default hover text ("trace 0, (43.2, 41.8)")
  wastes it. The saver warns when no trace has one.
- **Palette slots assigned in order**, never cycled or re-sorted. The palette is
  colour-vision-deficiency validated pairwise in that order; re-ordering silently breaks
  that property.
- **Explicit `width`/`height` via `figure_size`**, in inches. Once a PNG is committed at a
  given size, the surrounding prose assumes it — pinning the size is what stops a template
  default from reflowing your document later.

In [ ]:
lims = [0, 100]
fig = go.Figure()

for (group, subset), colour in zip(data.groupby("group"), CATEGORICAL):
    fig.add_trace(go.Scatter(
        x=subset["reference"], y=subset["measured"], mode="markers",
        name=f"group {group}",
        customdata=np.stack([subset["case"], subset["residual"]], axis=-1),
        marker=dict(size=8, color=colour, line=dict(color="#0b0b0b", width=0.3)),
        hovertemplate=("%{customdata[0]}<br>"
                       "reference %{x:.1f}<br>"
                       "measured %{y:.1f}<br>"
                       "residual %{customdata[1]:+.1f}<extra></extra>"),
    ))

fig.add_trace(go.Scatter(x=lims, y=lims, mode="lines", name="1:1", hoverinfo="skip",
                         line=dict(color="#4a4945", dash="dash", width=1)))

fig.update_layout(
    title="Measured against reference",
    xaxis_title="Reference value", yaxis_title="Measured value",
    xaxis_range=lims, yaxis_range=lims,
    legend=dict(x=0.02, y=0.98, xanchor="left", yanchor="top"),
    **figure_size(6.0, 4.5),
)

save_fig(fig, "example_scatter.png",
         hover_fields=["case", "reference", "measured", "residual"],
         caption="Synthetic measurements against their reference values.",
         label="example_scatter")
fig.show()

## The table

`save_table` renders a DataFrame as booktabs LaTeX into the chapter's `tables.tex`,
tagged so the same name replaces in place rather than accumulating. Pull it into the prose
with `\ExecuteMetaData[Chapters/Example/tables.tex]{example_summary}`.

In [ ]:
summary = (data.groupby("group")
           .agg(n=("case", "size"), mean=("measured", "mean"), bias=("residual", "mean"))
           .reset_index())

save_table(summary, "example_summary", decimals=2,
           caption="Synthetic cohort summary by group.")
summary.round(2)

## What just happened

If `DOC_REPO` is set and the config allows it, this run wrote:

| Where | What |
|---|---|
| `$DOC_REPO/Chapters/Example/Figures/example_scatter.png` | the PNG the document compiles |
| `figures_html/example_scatter.html` | the interactive export |
| `figures_html/figures_manifest.json` | an entry keyed by LaTeX label |
| `$DOC_REPO/Chapters/Example/tables.tex` | the tagged table block |

Two commands worth running now:

```bash
check-figure-parity --snapshot   # record these dimensions as the baseline to hold to
check-figure-parity --figures    # which saved figures the document actually renders
```

The second is the one that catches the quiet failure: a figure can be perfect on disk and
still be invisible in the PDF because nothing includes it.